# Multilingual Consensus Purification — Defending Zero-Shot VLMs Against Adversarial Attacks

**A thorough, runnable tutorial** implementing the pipeline from
*"Multilingual Consensus Purification as a Defense Against Adversarial Attacks on Zero-Shot
Vision–Language Models."*

> **Prerequisite:** the companion notebook `intro_to_CLIP.ipynb` explains how CLIP turns images
> and text into a shared embedding space. This notebook assumes that intuition and builds attacks
> and defenses on top of it.

### The core idea in one paragraph

A multilingual zero-shot model (multilingual CLIP) labels an image by matching its embedding to
**text labels in many languages**, all sharing **one image encoder**. An adversarial attack crafted
against *one* language's labels misaligns the shared embedding for *that* language while often
leaving the others correct — so **the languages disagree**. This tutorial tests whether that
disagreement can be turned into a defense, using three approaches:

1. a **multilingual ensemble** (average the per-language scores),
2. a **cross-lingual disagreement detector** (flag images where languages disagree), and
3. a **consensus-purification denoiser** — a small CNN trained *self-supervised* to restore
   cross-lingual agreement.

We then do the honest thing the paper insists on: evaluate the denoiser under an **adaptive attack
that backpropagates through it**, because purification defenses are known to be fragile.

### Roadmap

| Section | What we build |
|--------|----------------|
| 1 | Load multilingual CLIP + STL-10, set up multilingual labels |
| 2 | A differentiable pixel-space classification pipeline |
| 3 | Clean zero-shot accuracy per language |
| 4 | FGSM & PGD attacks; the **single-language attack** and cross-lingual **transfer** |
| 5 | **Adaptive multi-language** attack and the **attacker-cost curve** |
| 6 | Defense 1 — **multilingual ensemble** |
| 7 | Defense 2 — **disagreement detector** (ROC / AUC) |
| 8 | Defense 3 — **consensus-purification denoiser** (self-supervised training) |
| 9 | The crucial test — **attacking through the denoiser** |
| 10 | Putting the results together + discussion |

> **Compute.** A GPU is strongly recommended (Colab *Runtime → GPU*). Defaults use a small image
> subset and short PGD loops so the whole notebook runs in a few minutes; every knob is exposed in
> the `CFG` config so you can scale up for real numbers.


## 0. Setup

In [ ]:
%pip install -q open_clip_torch torch torchvision scikit-learn matplotlib


In [ ]:
import torch, torch.nn as nn, torch.nn.functional as F
import numpy as np, matplotlib.pyplot as plt
import open_clip
from torchvision import datasets, transforms as T
from torchvision.transforms import InterpolationMode, Normalize
from sklearn.metrics import roc_auc_score, roc_curve

device = "cuda" if torch.cuda.is_available() else "cpu"
torch.manual_seed(0); np.random.seed(0)
print("Device:", device, "| open_clip", open_clip.__version__)


### The whole pipeline at a glance

One image goes through **one shared encoder**, then gets scored against label sets in several
languages. If the languages **agree**, we trust the consensus; if a language-specific attack makes
them **disagree**, the three defenses kick in. Keep this map in mind as we build each piece.


In [ ]:
from matplotlib.patches import FancyBboxPatch, FancyArrowPatch

fig, ax = plt.subplots(figsize=(12, 7)); ax.axis("off")
ax.set_xlim(0, 12); ax.set_ylim(0, 7)

def _box(x, y, w, h, t, c, fs=9):
    ax.add_patch(FancyBboxPatch((x, y), w, h, boxstyle="round,pad=0.04",
                                fc=c, ec="black", lw=1.4))
    ax.text(x + w/2, y + h/2, t, ha="center", va="center", fontsize=fs, weight="bold")

def _arr(x1, y1, x2, y2, c="#555"):
    ax.add_patch(FancyArrowPatch((x1, y1), (x2, y2), arrowstyle="-|>",
                                 mutation_scale=14, lw=1.4, color=c))

_box(0.2, 5.4, 1.7, 1.0, "Input\nimage", "#cfe8ff")
_box(2.4, 5.4, 2.6, 1.0, "Shared image encoder\n(frozen M-CLIP ViT)", "#9ecbff")
_arr(1.9, 5.9, 2.4, 5.9)

langs5 = [("EN", "#ffd9b3"), ("KO", "#ffe9cc"), ("ES", "#ffd9b3"),
          ("FR", "#ffe9cc"), ("JA", "#ffd9b3")]
for i, (lg, c) in enumerate(langs5):
    y = 6.2 - i*1.05
    _box(5.7, y, 2.1, 0.85, f"{lg} labels: cosine", c, fs=8)
    _arr(5.0, 5.9, 5.7, y + 0.42)

_box(8.4, 3.4, 2.2, 1.0, "Per-language\npredictions", "#edd1ff")
for i in range(5):
    _arr(7.8, 6.2 - i*1.05 + 0.42, 8.4, 3.9)

_box(8.4, 1.4, 2.2, 1.3, "Agree -> consensus\n(likely clean)\nDisagree -> attack?", "#fff3b0", fs=8)
_arr(9.5, 3.4, 9.5, 2.7)

_box(0.2, 0.3, 2.7, 1.1, "Defense 1\nEnsemble (vote-average)", "#cdebc5", fs=8)
_box(3.2, 0.3, 2.7, 1.1, "Defense 2\nDisagreement detector", "#cdebc5", fs=8)
_box(6.2, 0.3, 2.7, 1.1, "Defense 3\nPurification denoiser", "#cdebc5", fs=8)
for x in (1.55, 4.55, 7.55):
    _arr(9.0, 1.4, x, 1.4, c="#888")

ax.set_title("Multilingual consensus pipeline: one encoder, many languages, three defenses",
             fontsize=12, weight="bold")
plt.tight_layout(); plt.show()


In [ ]:
# ---- Central configuration: tweak these to trade speed for fidelity ----
class CFG:
    model_name   = "xlm-roberta-base-ViT-B-32"     # multilingual CLIP: ONE image encoder,
    pretrained   = "laion5b_s13b_b90k"             #   multilingual text tower
    subset_size  = 128      # images used for evaluation (raise to 512+ for stable numbers)
    train_size   = 256      # images used to train the denoiser
    batch_size   = 32
    eps_list     = [2, 4, 8, 16]   # L-inf budgets, in /255
    pgd_steps    = 10
    pgd_alpha    = 1.0/255  # PGD step size (in [0,1] pixel units)
    denoiser_steps = 400    # optimizer steps for denoiser training
    denoiser_train_eps = 8  # /255, budget used to craft training adversarials
    denoiser_train_pgd_steps = 5
    fidelity_lambda = 0.05  # weight of the fidelity term that prevents collapse
CFG = CFG()
print({k:v for k,v in vars(CFG).items() if not k.startswith('_')})


## 1. Load multilingual CLIP, the data, and the multilingual labels

We use `xlm-roberta-base-ViT-B-32` from `open_clip`. Crucially, it pairs a single ViT-B/32 image
encoder with a **multilingual XLM-RoBERTa text tower**, so every language is scored against the
*same* image embedding — exactly the shared-encoder setting the paper requires.

Dataset: **STL-10**, chosen because its ten classes are concrete and translate cleanly
(airplane, bird, car, cat, deer, dog, horse, monkey, ship, truck).


In [ ]:
model, _, preprocess = open_clip.create_model_and_transforms(
    CFG.model_name, pretrained=CFG.pretrained)
tokenizer = open_clip.get_tokenizer(CFG.model_name)
model = model.to(device).eval()
for p in model.parameters():          # CLIP stays frozen throughout
    p.requires_grad_(False)
logit_scale = model.logit_scale.exp().item()
print(f"Loaded {CFG.model_name}. logit_scale = {logit_scale:.2f}")


In [ ]:
# Recover the model's expected input size and normalization so we can attack in
# clean [0,1] PIXEL space and apply normalization *inside* the forward pass.
def get_norm(preprocess):
    for t in preprocess.transforms:
        if isinstance(t, Normalize):
            return torch.tensor(t.mean), torch.tensor(t.std)
    # fallback to standard CLIP stats
    return (torch.tensor([0.48145466, 0.4578275, 0.40821073]),
            torch.tensor([0.26862954, 0.26130258, 0.27577711]))

img_size = model.visual.image_size
img_size = img_size[0] if isinstance(img_size, (tuple, list)) else img_size
MEAN, STD = get_norm(preprocess)
MEAN = MEAN.view(1, 3, 1, 1).to(device)
STD  = STD.view(1, 3, 1, 1).to(device)

# Preprocessing that stops at [0,1] (NO normalization) -> attack space
preprocess_pixels = T.Compose([
    T.Resize(img_size, interpolation=InterpolationMode.BICUBIC),
    T.CenterCrop(img_size),
    T.ToTensor(),               # produces values in [0,1]
])
print("Input size:", img_size, "| mean:", MEAN.flatten().tolist())


In [ ]:
# Load STL-10 test split and build a fixed random subset as [0,1] tensors.
stl = datasets.STL10(root="./data", split="test", download=True,
                     transform=preprocess_pixels)
CLASS_NAMES_EN = ['airplane','bird','car','cat','deer','dog','horse','monkey','ship','truck']

g = torch.Generator().manual_seed(0)
perm = torch.randperm(len(stl), generator=g)
eval_idx  = perm[:CFG.subset_size].tolist()
train_idx = perm[CFG.subset_size:CFG.subset_size + CFG.train_size].tolist()

def stack_subset(idxs):
    xs = torch.stack([stl[i][0] for i in idxs])
    ys = torch.tensor([stl[i][1] for i in idxs])
    return xs, ys

X_eval, y_eval = stack_subset(eval_idx)
X_train, y_train = stack_subset(train_idx)
print("eval:", tuple(X_eval.shape), "| train:", tuple(X_train.shape),
      "| pixel range:", (X_eval.min().item(), X_eval.max().item()))


In [ ]:
# Multilingual class names + a natural per-language prompt template.
LABELS = {
    "en": ['airplane','bird','car','cat','deer','dog','horse','monkey','ship','truck'],
    "ko": ['비행기','새','자동차','고양이','사슴','개','말','원숭이','배','트럭'],
    "es": ['avión','pájaro','coche','gato','ciervo','perro','caballo','mono','barco','camión'],
    "fr": ['avion','oiseau','voiture','chat','cerf','chien','cheval','singe','bateau','camion'],
    "ja": ['飛行機','鳥','車','猫','鹿','犬','馬','猿','船','トラック'],
}
TEMPLATES = {
    "en": "a photo of a {}",
    "ko": "{}의 사진",
    "es": "una foto de un {}",
    "fr": "une photo d'un {}",
    "ja": "{}の写真",
}
ALL_LANGS = list(LABELS.keys())
print("Languages:", ALL_LANGS)
for lg in ALL_LANGS:
    print(f"  {lg}: {TEMPLATES[lg].format(LABELS[lg][3])}")  # show the 'cat' prompt


## 2. A differentiable pixel-space classification pipeline

For adversarial attacks we must differentiate the loss **with respect to the pixels**. So we keep
images in `[0,1]` and fold the CLIP normalization into the forward pass. We also **cache** the
per-language text features once (they never change — the text tower is frozen and the prompts are
fixed), so attacks and defenses just reuse them.


In [ ]:
@torch.no_grad()
def encode_text_features(langs):
    # Returns {lang: (num_classes, dim) L2-normalized text features}, cached on device.
    feats = {}
    for lg in langs:
        prompts = [TEMPLATES[lg].format(c) for c in LABELS[lg]]
        tf = model.encode_text(tokenizer(prompts).to(device))
        feats[lg] = F.normalize(tf, dim=-1)
    return feats

TEXT_FEATS = encode_text_features(ALL_LANGS)
print({lg: tuple(v.shape) for lg, v in TEXT_FEATS.items()})


In [ ]:
def clip_image_features(images01):
    # images01: (B,3,H,W) in [0,1], differentiable. Returns L2-normalized image features.
    x = (images01 - MEAN) / STD
    feats = model.encode_image(x)
    return F.normalize(feats, dim=-1)

def logits_lang(images01, lang_feats):
    # Cosine-similarity logits for one language's cached text features.
    return logit_scale * clip_image_features(images01) @ lang_feats.T

@torch.no_grad()
def predict_lang(images01, lang, bs=64):
    preds = []
    for i in range(0, images01.size(0), bs):
        xb = images01[i:i+bs].to(device)
        preds.append(logits_lang(xb, TEXT_FEATS[lang]).argmax(-1).cpu())
    return torch.cat(preds)

@torch.no_grad()
def accuracy_lang(images01, labels, lang):
    return (predict_lang(images01, lang) == labels).float().mean().item()


## 3. Clean zero-shot accuracy per language

Before any attack, how well does each language classify the clean images? Languages will differ a
bit (the text tower is imperfect across scripts), but all should be well above the 10% chance line.
This is our baseline.


In [ ]:
clean_acc = {lg: accuracy_lang(X_eval, y_eval, lg) for lg in ALL_LANGS}
for lg, a in clean_acc.items():
    print(f"  {lg}: {a:.1%}")

plt.figure(figsize=(6,3))
plt.bar(clean_acc.keys(), [v*100 for v in clean_acc.values()], color="tab:green")
plt.axhline(10, ls="--", c="gray", label="chance (10%)")
plt.ylabel("clean accuracy (%)"); plt.title("Clean zero-shot accuracy by language")
plt.legend(); plt.tight_layout(); plt.show()


## 4. Attacks: FGSM, PGD, and the single-language transfer experiment

Attacks are **white-box**, per-image, and **$L_\infty$-bounded**: the perturbation $\delta$
satisfies $\lVert\delta\rVert_\infty \le \varepsilon$ and the image stays in $[0,1]$.

- **FGSM** (Goodfellow et al., 2015): one step of size $\varepsilon$ along the sign of the gradient.
- **PGD** (Madry et al., 2018): iterated FGSM with a random start and projection back into the
  $\varepsilon$-ball — a much stronger attack.

The **single-language attack** maximizes the cross-entropy loss on **one** language's labels
(English). We then measure how far the resulting perturbation **transfers** to the *other*
languages. The paper's hypothesis **H1** is that transfer is only *partial* and grows with
$\varepsilon$ and with language closeness — that partial transfer is the disagreement the defenses
will exploit.


In [ ]:
def fgsm(images, labels, feats_list, eps):
    # feats_list: list of language feature tensors; loss summed over them.
    images = images.clone().detach().to(device).requires_grad_(True)
    labels = labels.to(device)
    loss = sum(F.cross_entropy(logits_lang(images, tf), labels) for tf in feats_list)
    grad, = torch.autograd.grad(loss, images)
    adv = (images + eps * grad.sign()).clamp(0, 1)
    return adv.detach()

def pgd(images, labels, feats_list, eps, alpha=None, steps=None, random_start=True):
    alpha = alpha or CFG.pgd_alpha; steps = steps or CFG.pgd_steps
    images = images.to(device); labels = labels.to(device)
    ori = images.clone().detach()
    if random_start:
        adv = (ori + torch.empty_like(ori).uniform_(-eps, eps)).clamp(0, 1).detach()
    else:
        adv = ori.clone().detach()
    for _ in range(steps):
        adv.requires_grad_(True)
        loss = sum(F.cross_entropy(logits_lang(adv, tf), labels) for tf in feats_list)
        grad, = torch.autograd.grad(loss, adv)
        adv = adv.detach() + alpha * grad.sign()
        adv = torch.min(torch.max(adv, ori - eps), ori + eps).clamp(0, 1).detach()
    return adv

def attack_in_batches(attack_fn, X, y, feats_list, eps, bs=None):
    # Generic batched driver returning a CPU tensor of adversarial images.
    bs = bs or CFG.batch_size
    outs = []
    for i in range(0, X.size(0), bs):
        adv = attack_fn(X[i:i+bs], y[i:i+bs], feats_list, eps)
        outs.append(adv.cpu())
    return torch.cat(outs)


In [ ]:
# Single-language (English) PGD attack at each epsilon; then evaluate EVERY language on the
# English-crafted adversarials to measure transfer.
transfer = {lg: [] for lg in ALL_LANGS}
adv_by_eps = {}
for e255 in CFG.eps_list:
    eps = e255 / 255.0
    X_adv = attack_in_batches(pgd, X_eval, y_eval, [TEXT_FEATS["en"]], eps)
    adv_by_eps[e255] = X_adv
    for lg in ALL_LANGS:
        transfer[lg].append(accuracy_lang(X_adv, y_eval, lg))
    print(f"eps={e255}/255  " +
          "  ".join(f"{lg}:{transfer[lg][-1]:.0%}" for lg in ALL_LANGS))


In [ ]:
# What does the attack actually look like? Same image, growing epsilon. The perturbation
# stays near-imperceptible on the image yet flips the prediction (note: panels are amplified).
i = 0
clean = X_eval[i]
fig, axes = plt.subplots(2, len(CFG.eps_list), figsize=(2.5*len(CFG.eps_list), 4.8))
for j, e255 in enumerate(CFG.eps_list):
    adv = adv_by_eps[e255][i]
    delta = (adv - clean).abs()
    axes[0, j].imshow(adv.permute(1, 2, 0).clamp(0, 1).numpy())
    axes[0, j].set_title(f"adv  eps={e255}/255", fontsize=9); axes[0, j].axis("off")
    axes[1, j].imshow((delta / (delta.max() + 1e-8)).permute(1, 2, 0).numpy())
    axes[1, j].set_title("|perturbation| (amplified)", fontsize=8); axes[1, j].axis("off")
axes[0, 0].set_ylabel("adversarial", fontsize=10)
fig.suptitle(f"Single-language attack on '{CLASS_NAMES_EN[y_eval[i]]}': larger eps, stronger noise",
             weight="bold")
plt.tight_layout(); plt.show()


In [ ]:
# Robust accuracy vs eps. English (the TARGET) collapses; other languages degrade only partly:
# that gap is the cross-lingual disagreement the defenses exploit.
plt.figure(figsize=(7,4))
for lg in ALL_LANGS:
    style = "o-" if lg == "en" else ".--"
    plt.plot(CFG.eps_list, [a*100 for a in transfer[lg]], style, label=lg,
             lw=2.5 if lg=="en" else 1.5)
plt.xlabel("epsilon (/255)"); plt.ylabel("accuracy on English-crafted adv (%)")
plt.title("Single-language (English) attack: transfer across languages")
plt.legend(title="evaluated in"); plt.grid(alpha=0.3); plt.tight_layout(); plt.show()


In [ ]:
# The same numbers as a heatmap (rows = language we evaluate in, columns = epsilon).
# Green = still robust, red = fooled. English (top row) reddens fastest; the distant
# languages stay greener — that is the disagreement signal, made visual.
A = np.array([[transfer[lg][k] for k in range(len(CFG.eps_list))] for lg in ALL_LANGS])
fig, ax = plt.subplots(figsize=(7, 3.6))
im = ax.imshow(A, cmap="RdYlGn", vmin=0, vmax=1, aspect="auto")
ax.set_xticks(range(len(CFG.eps_list))); ax.set_xticklabels([f"{e}/255" for e in CFG.eps_list])
ax.set_yticks(range(len(ALL_LANGS))); ax.set_yticklabels(ALL_LANGS)
ax.set_xlabel("attack budget (epsilon)"); ax.set_ylabel("evaluated in language")
for i in range(len(ALL_LANGS)):
    for j in range(len(CFG.eps_list)):
        ax.text(j, i, f"{A[i,j]:.0%}", ha="center", va="center", fontsize=8)
plt.colorbar(im, label="robust accuracy")
ax.set_title("Robust accuracy under the English-only attack")
plt.tight_layout(); plt.show()


In [ ]:
# Quantify TRANSFER FRACTION: of images that a non-target language got right on clean,
# what fraction does the English attack flip? (1.0 = full transfer, 0.0 = no transfer)
def transfer_fraction(X_adv, lang):
    clean_pred = predict_lang(X_eval, lang)
    adv_pred   = predict_lang(X_adv,  lang)
    correct_clean = clean_pred == y_eval
    flipped = (adv_pred != y_eval) & correct_clean
    return flipped.sum().item() / max(1, correct_clean.sum().item())

print("Transfer fraction of the English attack into other languages:")
for e255 in CFG.eps_list:
    row = {lg: round(transfer_fraction(adv_by_eps[e255], lg), 2)
           for lg in ALL_LANGS if lg != "en"}
    print(f"  eps={e255}/255 -> {row}")


### The core mechanism, made visual: *the languages disagree*

This is the single most important picture in the notebook. We take images that the English attack
**flipped**, and show what **each language** predicts on that *same* adversarial image. English is
fooled (red); the other languages frequently still get it right (green). That split is precisely
what the ensemble, the detector, and the denoiser exploit.


In [ ]:
e255 = CFG.eps_list[-1]                 # strongest attack -> clearest disagreement
Xadv = adv_by_eps[e255]
en_clean = predict_lang(X_eval, "en"); en_adv = predict_lang(Xadv, "en")
flipped = (((en_clean == y_eval) & (en_adv != y_eval)).nonzero(as_tuple=True)[0]).tolist()[:6]
preds_adv = {lg: predict_lang(Xadv, lg) for lg in ALL_LANGS}

if len(flipped) == 0:
    print("No English flips at this epsilon/subset — raise eps or subset_size to see disagreement.")
else:
    n = len(flipped)
    fig, axes = plt.subplots(n, len(ALL_LANGS) + 1,
                             figsize=(1.6*(len(ALL_LANGS)+1), 1.8*n), squeeze=False)
    for r, idx in enumerate(flipped):
        true_c = y_eval[idx].item()
        axes[r, 0].imshow(Xadv[idx].permute(1, 2, 0).clamp(0, 1).numpy())
        axes[r, 0].set_title(f"adv image\ntrue: {CLASS_NAMES_EN[true_c]}", fontsize=8)
        axes[r, 0].axis("off")
        for c, lg in enumerate(ALL_LANGS):
            p = preds_adv[lg][idx].item(); ok = (p == true_c)
            ax = axes[r, c+1]
            ax.set_facecolor("#cdebc5" if ok else "#f6c6c6")
            ax.text(0.5, 0.5, f"{lg.upper()}\n{CLASS_NAMES_EN[p]}\n{'OK' if ok else 'FOOLED'}",
                    ha="center", va="center", fontsize=8,
                    weight="bold" if not ok else "normal")
            ax.set_xticks([]); ax.set_yticks([])
    fig.suptitle(f"One adversarial image, five languages (eps={e255}/255):  "
                 f"green = correct, red = fooled", weight="bold", y=1.01)
    plt.tight_layout(); plt.show()


### Where the attack pushes the embedding

Because all languages share **one** image encoder, the attack moves a single image embedding. We
project clean and adversarial image embeddings into 2D (PCA) and draw an arrow for each image. The
systematic drift shows the attack nudging embeddings across decision boundaries.


In [ ]:
sel = list(range(min(80, X_eval.size(0))))
e255 = 8
with torch.no_grad():
    f_clean = clip_image_features(X_eval[sel].to(device)).cpu().numpy()
    f_adv   = clip_image_features(adv_by_eps[e255][sel].to(device)).cpu().numpy()

both = np.concatenate([f_clean, f_adv]); both = both - both.mean(0, keepdims=True)
_, _, Vt = np.linalg.svd(both, full_matrices=False)
C = both @ Vt[:2].T
cc, ca = C[:len(sel)], C[len(sel):]
labs = y_eval[sel].numpy()

fig, ax = plt.subplots(figsize=(7.5, 6))
for k in range(len(sel)):
    ax.annotate("", ca[k], cc[k],
                arrowprops=dict(arrowstyle="->", color="gray", alpha=0.35, lw=0.7))
sc = ax.scatter(cc[:, 0], cc[:, 1], c=labs, cmap="tab10", s=45, vmin=0, vmax=9)
ax.scatter(ca[:, 0], ca[:, 1], c=labs, cmap="tab10", s=55, marker="x", vmin=0, vmax=9)
ax.set_title(f"Attack displaces image embeddings (eps={e255}/255)\n"
             "dot = clean,  ✕ = adversarial,  color = true class,  arrow = clean→adv")
ax.set_xlabel("PC 1"); ax.set_ylabel("PC 2")
cb = plt.colorbar(sc, ticks=range(10)); cb.ax.set_yticklabels(CLASS_NAMES_EN); cb.set_label("class")
plt.tight_layout(); plt.show()


**Reading the result.** English robust accuracy should drop sharply while Korean, Spanish,
French and Japanese hold up far better — and the transfer fraction into other languages stays well
below 1.0. Typically transfer is *higher* into languages closer to English (the Romance languages)
than into Korean or Japanese, matching hypothesis **H1**. The persistent disagreement is what makes
the next three defenses possible.


## 5. Adaptive multi-language attack & the attacker-cost curve

A smart attacker won't stop at one language. The **multi-language attack** maximizes the *summed*
loss over several languages at once, forcing them to agree on the *wrong* class. Hypothesis **H2**:
the attacker's budget grows as more languages must be fooled simultaneously.

We trace the **attacker-cost curve** — the minimum $\varepsilon$ needed to drive the multilingual
**ensemble** (defined fully in §6) below a target accuracy, as a function of how many languages the
attacker targets.


In [ ]:
def ensemble_logits(images01, langs):
    # Softmax-average the per-language probabilities -> ensemble (log) scores.
    probs = 0
    for lg in langs:
        probs = probs + logits_lang(images01, TEXT_FEATS[lg]).softmax(-1)
    return probs / len(langs)

@torch.no_grad()
def ensemble_accuracy(images01, labels, langs, bs=64):
    preds = []
    for i in range(0, images01.size(0), bs):
        xb = images01[i:i+bs].to(device)
        preds.append(ensemble_logits(xb, langs).argmax(-1).cpu())
    return (torch.cat(preds) == labels).float().mean().item()


In [ ]:
# Targeting the first k languages, find the smallest eps that pushes ENSEMBLE accuracy
# (over ALL languages) below `target_acc`. Finer eps grid -> smoother curve (slower).
ATTACK_SETS = [ALL_LANGS[:k] for k in range(1, len(ALL_LANGS)+1)]
eps_grid = [1,2,3,4,6,8,12,16,24,32]      # /255
target_acc = 0.30

cost_curve = []
for langs in ATTACK_SETS:
    min_eps = None
    for e255 in eps_grid:
        X_adv = attack_in_batches(pgd, X_eval, y_eval,
                                  [TEXT_FEATS[l] for l in langs], e255/255.0)
        acc = ensemble_accuracy(X_adv, y_eval, ALL_LANGS)
        if acc <= target_acc:
            min_eps = e255; break
    cost_curve.append(min_eps if min_eps is not None else eps_grid[-1])
    print(f"target langs={langs} -> min eps to break ensemble: "
          f"{cost_curve[-1]}{'+' if min_eps is None else ''}/255")


In [ ]:
plt.figure(figsize=(6,4))
plt.plot(range(1, len(ATTACK_SETS)+1), cost_curve, "o-", color="tab:red")
plt.xlabel("# languages attacked simultaneously")
plt.ylabel(f"min eps (/255) to push ensemble ≤ {target_acc:.0%}")
plt.title("Attacker-cost curve (H2)")
plt.grid(alpha=0.3); plt.tight_layout(); plt.show()


**Reading the result.** The curve should trend **upward**: fooling more languages at once
costs more budget. (With a tiny subset the curve can be noisy — raise `subset_size` and use a finer
`eps_grid` for a clean monotone trend.) This rising cost is the quantitative payoff of the
multilingual design.


## 6. Defense 1 — the multilingual ensemble

The simplest defense is **training-free**: softmax-average the per-language scores and take the
argmax (we already defined `ensemble_logits`). Because a single-language attack leaves most
languages correct, the averaged vote is hard to flip. Let's compare per-language robust accuracy to
the ensemble under the **single-language English** attack.


In [ ]:
ens_acc = []
for e255 in CFG.eps_list:
    ens_acc.append(ensemble_accuracy(adv_by_eps[e255], y_eval, ALL_LANGS))

plt.figure(figsize=(7,4))
plt.plot(CFG.eps_list, [a*100 for a in transfer["en"]], "o-", c="tab:blue",
         label="English only (attacked)")
plt.plot(CFG.eps_list, [a*100 for a in ens_acc], "s-", c="tab:green",
         label="multilingual ensemble")
plt.xlabel("epsilon (/255)"); plt.ylabel("robust accuracy (%)")
plt.title("Ensemble defense vs the single-language attack")
plt.legend(); plt.grid(alpha=0.3); plt.tight_layout(); plt.show()
print("Ensemble robust accuracy:",
      {e: round(a,3) for e,a in zip(CFG.eps_list, ens_acc)})


## 7. Defense 2 — the cross-lingual disagreement *detector*

Instead of trying to *correct* the prediction, we can **flag** suspicious images. Define a
**disagreement score**: how much do the per-language predictions disagree? Clean images should
produce broad agreement (low score); attacked images should produce disagreement (high score).

We use one minus the modal-vote fraction across languages, and evaluate it as a binary detector
(clean vs adversarial) with an **ROC curve** and its **AUC**.


In [ ]:
@torch.no_grad()
def disagreement_score(images01, langs=ALL_LANGS, bs=64):
    # For each image: 1 - (votes for the modal class) / (#languages). 0 = full agreement.
    scores = []
    for i in range(0, images01.size(0), bs):
        xb = images01[i:i+bs].to(device)
        votes = torch.stack([logits_lang(xb, TEXT_FEATS[lg]).argmax(-1) for lg in langs], 1)
        # modal count per row
        modal = []
        for row in votes:
            modal.append(torch.bincount(row, minlength=len(LABELS["en"])).max().item())
        modal = torch.tensor(modal, device=xb.device)
        scores.append((1.0 - modal.float()/len(langs)).cpu())
    return torch.cat(scores)

# Build a clean-vs-adversarial detection set at a representative epsilon.
det_eps = 8
clean_scores = disagreement_score(X_eval)
adv_scores   = disagreement_score(adv_by_eps[det_eps])

y_true  = np.r_[np.zeros(len(clean_scores)), np.ones(len(adv_scores))]
y_score = np.r_[clean_scores.numpy(), adv_scores.numpy()]
auc = roc_auc_score(y_true, y_score)
fpr, tpr, _ = roc_curve(y_true, y_score)
print(f"Disagreement detector AUC at eps={det_eps}/255: {auc:.3f}")


In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(11,4))
ax[0].hist(clean_scores.numpy(), bins=12, alpha=0.6, label="clean", color="tab:green")
ax[0].hist(adv_scores.numpy(),   bins=12, alpha=0.6, label="adversarial", color="tab:red")
ax[0].set_xlabel("disagreement score"); ax[0].set_ylabel("count")
ax[0].set_title("Score distributions"); ax[0].legend()

ax[1].plot(fpr, tpr, lw=2, label=f"AUC = {auc:.3f}")
ax[1].plot([0,1],[0,1],"--",c="gray")
ax[1].set_xlabel("false positive rate"); ax[1].set_ylabel("true positive rate")
ax[1].set_title(f"Detector ROC (eps={det_eps}/255)"); ax[1].legend()
plt.tight_layout(); plt.show()


**Reading the result.** An AUC well above 0.5 means cross-lingual disagreement is a usable
adversarial-detection signal. The detector doesn't *fix* predictions — it abstains/flags — which is
useful when a wrong answer is costlier than a "please review" flag. Note it only catches
*language-specific* attacks; a *language-agnostic* attack that preserves agreement (out of scope
here) would slip past, by construction.


## 8. Defense 3 — the consensus-purification denoiser (the main method)

Now the central contribution: a small **residual CNN** $g$ that **purifies** an image before
classification. It is trained **self-supervised** — no human labels:

- For a clean image, the multilingual **ensemble's** prediction is treated as a free **pseudo-label**
  (the "clean consensus").
- We craft an adversarial version, purify it with $g$, and train $g$ so the **purified** image's
  ensemble prediction matches that clean-consensus pseudo-label.
- A **fidelity term** $\lambda\lVert g(x)-x\rVert^2$ keeps $g$ from collapsing to a trivial
  constant/blurred output that satisfies consensus without preserving the image.

At test time: **purify, then classify**.


In [ ]:
class Denoiser(nn.Module):
    # Lightweight residual CNN operating on [0,1] images; output = clamp(x + residual).
    def __init__(self, ch=48):
        super().__init__()
        self.body = nn.Sequential(
            nn.Conv2d(3, ch, 3, padding=1), nn.GroupNorm(8, ch), nn.ReLU(inplace=True),
            nn.Conv2d(ch, ch, 3, padding=1), nn.GroupNorm(8, ch), nn.ReLU(inplace=True),
            nn.Conv2d(ch, ch, 3, padding=1), nn.GroupNorm(8, ch), nn.ReLU(inplace=True),
            nn.Conv2d(ch, 3, 3, padding=1),
        )
        nn.init.zeros_(self.body[-1].weight); nn.init.zeros_(self.body[-1].bias)  # start ≈ identity
    def forward(self, x):
        return (x + self.body(x)).clamp(0, 1)

denoiser = Denoiser().to(device)
print("Denoiser params:", sum(p.numel() for p in denoiser.parameters()))


In [ ]:
@torch.no_grad()
def consensus_pseudolabels(images01, langs=ALL_LANGS, bs=64):
    # Free labels: the clean multilingual ensemble's argmax.
    out = []
    for i in range(0, images01.size(0), bs):
        xb = images01[i:i+bs].to(device)
        out.append(ensemble_logits(xb, langs).argmax(-1).cpu())
    return torch.cat(out)

# Precompute pseudo-labels for the (label-free) training pool.
train_pseudo = consensus_pseudolabels(X_train)
print("Pseudo-label agreement with true labels (sanity check, not used in training):",
      (train_pseudo == y_train).float().mean().item())


In [ ]:
def train_denoiser(denoiser, X_pool, pseudo, cfg):
    opt = torch.optim.Adam(denoiser.parameters(), lr=2e-4)
    n = X_pool.size(0); bs = cfg.batch_size
    eps = cfg.denoiser_train_eps / 255.0
    denoiser.train()
    hist = []
    for step in range(cfg.denoiser_steps):
        idx = torch.randint(0, n, (bs,))
        x = X_pool[idx].to(device); pl = pseudo[idx].to(device)

        # 1) craft a language-specific adversarial (English) against the FROZEN clip.
        #    The denoiser is NOT in this loop -> non-adaptive training adversarials.
        x_adv = pgd(x, pl, [TEXT_FEATS["en"]], eps,
                    alpha=cfg.pgd_alpha, steps=cfg.denoiser_train_pgd_steps).detach()

        # 2) purify and push the ensemble prediction back to the clean consensus.
        x_pur = denoiser(x_adv)
        ens = torch.log(ensemble_logits(x_pur, ALL_LANGS) + 1e-8)   # log-probs for NLL
        loss_cons = F.nll_loss(ens, pl)
        loss_fid  = F.mse_loss(x_pur, x_adv)                        # prevents collapse
        loss = loss_cons + cfg.fidelity_lambda * loss_fid

        opt.zero_grad(); loss.backward(); opt.step()
        hist.append((loss_cons.item(), loss_fid.item()))
        if (step+1) % max(1, cfg.denoiser_steps//10) == 0:
            print(f"step {step+1:4d}/{cfg.denoiser_steps}  "
                  f"consensus={loss_cons.item():.3f}  fidelity={loss_fid.item():.4f}")
    denoiser.eval()
    return np.array(hist)

hist = train_denoiser(denoiser, X_train, train_pseudo, CFG)


In [ ]:
plt.figure(figsize=(6,3))
plt.plot(hist[:,0], label="consensus loss")
plt.plot(hist[:,1]*10, label="fidelity loss (×10)")
plt.xlabel("step"); plt.ylabel("loss"); plt.legend()
plt.title("Denoiser training"); plt.tight_layout(); plt.show()


In [ ]:
# Evaluate the denoiser NON-ADAPTIVELY: attack the clip, then purify, then classify.
@torch.no_grad()
def purify(images01, bs=64):
    outs = []
    for i in range(0, images01.size(0), bs):
        outs.append(denoiser(images01[i:i+bs].to(device)).cpu())
    return torch.cat(outs)

rows = []
for e255 in CFG.eps_list:
    Xadv = adv_by_eps[e255]
    acc_no  = ensemble_accuracy(Xadv,         y_eval, ALL_LANGS)   # ensemble, no denoiser
    acc_pur = ensemble_accuracy(purify(Xadv), y_eval, ALL_LANGS)   # ensemble + denoiser
    rows.append((e255, acc_no, acc_pur))
    print(f"eps={e255}/255  ensemble={acc_no:.1%}   ensemble+denoiser={acc_pur:.1%}")


In [ ]:
# Visual sanity check: clean / adversarial / purified for one image.
def to_img(t): return t.permute(1,2,0).clamp(0,1).cpu().numpy()
i = 0; e255 = 16
clean_i = X_eval[i]; adv_i = adv_by_eps[e255][i]
pur_i = denoiser(adv_i.unsqueeze(0).to(device)).squeeze(0).detach()
fig, ax = plt.subplots(1, 4, figsize=(13,3.4))
ax[0].imshow(to_img(clean_i)); ax[0].set_title(f"clean\n(true: {CLASS_NAMES_EN[y_eval[i]]})")
ax[1].imshow(to_img(adv_i));   ax[1].set_title(f"adversarial\neps={e255}/255")
ax[2].imshow(to_img(pur_i));   ax[2].set_title("purified")
ax[3].imshow(to_img((adv_i-clean_i).abs()/(2*e255/255)+0.0)); ax[3].set_title("|perturbation|\n(scaled)")
for a in ax: a.axis("off")
plt.tight_layout(); plt.show()


**Reading the result.** Non-adaptively, the denoiser should **recover a good chunk** of
accuracy over the already-strong ensemble (hypothesis **H3**, first half). But this is *not* the
real test — the attacker in §9 knows the denoiser exists.


## 9. The crucial test — attacking *through* the denoiser

Athalye, Carlini & Wagner (2018) showed that many purification defenses only *look* robust because
they obscure gradients; a proper **adaptive** attacker that differentiates through the defense
breaks them. Since our denoiser is fully differentiable, the adaptive attack is just PGD on the
**composition** $\text{classify}(g(x))$ — no approximation needed.

Hypothesis **H3** (second half): under this through-the-denoiser attack, the denoiser's benefit may
shrink, possibly to little. Measuring that honestly is the point.


In [ ]:
def pgd_through_denoiser(images, labels, feats_list, eps, alpha=None, steps=None):
    alpha = alpha or CFG.pgd_alpha; steps = steps or CFG.pgd_steps
    images = images.to(device); labels = labels.to(device)
    ori = images.clone().detach()
    adv = (ori + torch.empty_like(ori).uniform_(-eps, eps)).clamp(0,1).detach()
    for _ in range(steps):
        adv.requires_grad_(True)
        x_pur = denoiser(adv)                                  # attack sees the denoiser
        loss = sum(F.cross_entropy(logits_lang(x_pur, tf), labels) for tf in feats_list)
        grad, = torch.autograd.grad(loss, adv)
        adv = adv.detach() + alpha * grad.sign()
        adv = torch.min(torch.max(adv, ori-eps), ori+eps).clamp(0,1).detach()
    return adv

def attack_through_in_batches(X, y, feats_list, eps, bs=None):
    bs = bs or CFG.batch_size; outs=[]
    for i in range(0, X.size(0), bs):
        outs.append(pgd_through_denoiser(X[i:i+bs], y[i:i+bs], feats_list, eps).cpu())
    return torch.cat(outs)


In [ ]:
# Compare three regimes at each epsilon, all measured with the ensemble classifier:
#   (a) ensemble only (no denoiser)              -> baseline defense
#   (b) non-adaptive  : attack clip, then purify -> looks great
#   (c) adaptive      : attack THROUGH denoiser   -> the honest number
adaptive_rows = []
for e255 in CFG.eps_list:
    eps = e255/255.0
    a_ens = ensemble_accuracy(adv_by_eps[e255], y_eval, ALL_LANGS)
    a_non = ensemble_accuracy(purify(adv_by_eps[e255]), y_eval, ALL_LANGS)
    X_thr = attack_through_in_batches(X_eval, y_eval, [TEXT_FEATS["en"]], eps)
    a_adp = ensemble_accuracy(purify(X_thr), y_eval, ALL_LANGS)
    adaptive_rows.append((e255, a_ens, a_non, a_adp))
    print(f"eps={e255}/255  ensemble={a_ens:.1%}  +denoiser(non-adaptive)={a_non:.1%}  "
          f"+denoiser(ADAPTIVE)={a_adp:.1%}")


In [ ]:
eps_x = [r[0] for r in adaptive_rows]
plt.figure(figsize=(7.5,4.5))
plt.plot(eps_x, [r[1]*100 for r in adaptive_rows], "o-", label="ensemble only")
plt.plot(eps_x, [r[2]*100 for r in adaptive_rows], "s-", label="+ denoiser (non-adaptive)")
plt.plot(eps_x, [r[3]*100 for r in adaptive_rows], "^--", c="tab:red",
         label="+ denoiser (adaptive / through-the-denoiser)")
plt.xlabel("epsilon (/255)"); plt.ylabel("robust accuracy (%)")
plt.title("The honest adaptive evaluation (H3)")
plt.legend(); plt.grid(alpha=0.3); plt.tight_layout(); plt.show()


**Reading the result.** Expect the non-adaptive curve to look strong and the **adaptive**
curve to fall toward (or below) the plain ensemble — quantifying how much of the denoiser's apparent
robustness was real versus an artifact of gradient obfuscation. A defense is only as good as its
*adaptive* number; reporting it is the paper's central methodological commitment.


## 10. Putting it together & discussion

In [ ]:
print("="*64)
print("SUMMARY (illustrative; depends on subset size, seed, and compute budget)")
print("="*64)
print(f"\nClean accuracy per language: " +
      ", ".join(f"{lg}={clean_acc[lg]:.0%}" for lg in ALL_LANGS))
print("\nUnder single-language (English) PGD attack:")
print(f"  {'eps':>6} | {'EN only':>8} | {'ensemble':>9} | {'+denoiser':>10} | {'+den ADAPT':>11}")
for (e255,a_ens,a_non,a_adp) in adaptive_rows:
    print(f"  {e255:>4}/255 | {transfer['en'][CFG.eps_list.index(e255)]:>7.0%} | "
          f"{a_ens:>8.0%} | {a_non:>9.0%} | {a_adp:>10.0%}")
print(f"\nDisagreement detector AUC (eps={det_eps}/255): {auc:.3f}")
print(f"Attacker-cost curve (min eps/255 vs #langs): {cost_curve}")


### What we built and what it shows

- **Single-language attacks transfer only partially** across languages (H1) — the disagreement
  signal exists and is stronger for distant languages like Korean/Japanese.
- **Attacking more languages costs more budget** (H2) — the attacker-cost curve trends upward.
- The **ensemble** and **disagreement detector** are training-free and genuinely help against
  *language-specific* attacks.
- The **consensus-purification denoiser** recovers accuracy **non-adaptively**, but the **adaptive,
  through-the-denoiser attack** is the number that matters — and it typically erodes much of that
  benefit (H3), consistent with the known fragility of purification defenses.

### Scope and honesty (straight from the paper)

- **Language-agnostic attacks are out of scope.** An attack that corrupts the shared image
  embedding directly preserves cross-lingual *agreement* (on the wrong class), so disagreement-based
  defenses cannot see it by construction.
- These notebook numbers are **illustrative** — small subset, short PGD, one seed. For
  publishable claims, raise `subset_size`, `pgd_steps`, and the eps grid; average over seeds; and
  add stronger adaptive attacks (more steps, restarts, EOT) before trusting any robustness number.

### Ideas to extend

1. **Prompt ensembling per language** (average several templates) for higher clean accuracy.
2. **Stronger adaptive attacks**: more PGD steps, multiple random restarts, Expectation-over-
   Transformation if you add randomized purification.
3. **More/typologically diverse languages** to probe how disagreement scales with language distance.
4. **A diffusion-based purifier** (Nie et al., 2022) in place of the residual CNN, evaluated with
   the same adaptive rigor.
5. **Higher-resolution data** (Imagenette) and a larger backbone to test whether the effects persist.

### References

- Radford et al. (2021) — CLIP. Goodfellow et al. (2015) — FGSM. Madry et al. (2018) — PGD.
- Athalye, Carlini & Wagner (2018) — obfuscated gradients (why purification needs adaptive eval).
- Nie et al. (2022) — diffusion-based adversarial purification.
- Multilingual-CLIP (M-CLIP) / multilingual CLIP via OpenCLIP (`xlm-roberta-*-ViT-*`).
